# Modelo de Predicción de Octavos de Final - Mundial 2026

**Versión limpia para presentación universitaria**

Este notebook deja únicamente las partes necesarias para trabajar la fase de **octavos de final**:

1. Carga de datos.
2. Limpieza básica.
3. Entrenamiento del modelo Poisson con partidos históricos.
4. Uso del fixture actualizado del Mundial 2026.
5. Respeto de resultados reales ya jugados.
6. Predicción únicamente de partidos pendientes.
7. Exportación de resultados finales.

> Esta versión no usa bloques antiguos de dieciseisavos simulados. Los partidos ya jugados se toman del CSV actualizado.

## 1. Importación de librerías

In [7]:
import os
import re
import math
import unicodedata
import warnings
from collections import defaultdict
from math import lgamma, log

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 2. Detección y carga de archivos

El notebook busca automáticamente los archivos principales. Si están dentro de una carpeta `data`, también los detecta.

In [8]:
# ============================================================
# 2. DETECCIÓN Y CARGA DE ARCHIVOS
# ============================================================

FILE_CANDIDATES = {
    "matches": [
        "matches_1930_2022.csv",
        "matches.csv",
        "partidos_historicos.csv"
    ],
    "fifa_2022": [
        "fifa_ranking_2022-10-06.csv",
        "Clasificación_FIFA_2022-10-06.csv",
        "Clasificacion_FIFA_2022-10-06.csv"
    ],
    "fifa_2026": [
        "fifa_ranking_2026-06-08.csv",
        "ranking_fifa_2026.csv",
        "ranking_fifa_actualizado.csv"
    ],
    "schedule": [
        "schedule_2026_actualizado.csv",
        "world-cup_2026.csv",
        "world_cup_2026.csv",
        "schedule_2026.csv"
    ],
    "world_cup": [
        "world_cup.csv",
        "Copa del Mundo.csv",
        "copa_del_mundo.csv"
    ]
}

SEARCH_DIRS = [".", "data", "datos", "/content", "/content/data"]

if "RUTA_DATOS" in globals():
    SEARCH_DIRS.insert(0, RUTA_DATOS)


def find_file(candidates, required=True):
    for folder in SEARCH_DIRS:
        for name in candidates:
            path = os.path.join(folder, name)
            if os.path.exists(path):
                return path
    if required:
        raise FileNotFoundError(
            "No encontré ninguno de estos archivos: " + str(candidates) +
            "\nSube el archivo a Colab o colócalo en la carpeta data/."
        )
    return None


def read_csv_robust(path):
    encodings = ["utf-8", "utf-8-sig", "latin1", "ISO-8859-1"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as error:
            last_error = error
    raise last_error

paths = {
    key: find_file(value, required=(key in ["matches", "fifa_2026", "schedule"]))
    for key, value in FILE_CANDIDATES.items()
}

print("Archivos detectados:")
for key, value in paths.items():
    print(f"- {key}: {value}")

matches = read_csv_robust(paths["matches"])
fifa_2026 = read_csv_robust(paths["fifa_2026"])
schedule_2026 = read_csv_robust(paths["schedule"])

fifa_2022 = read_csv_robust(paths["fifa_2022"]) if paths["fifa_2022"] else None
world_cup = read_csv_robust(paths["world_cup"]) if paths["world_cup"] else None

print("\nDatasets cargados:")
print("matches:", matches.shape)
print("fifa_2026:", fifa_2026.shape)
print("schedule_2026:", schedule_2026.shape)
if fifa_2022 is not None:
    print("fifa_2022:", fifa_2022.shape)
if world_cup is not None:
    print("world_cup:", world_cup.shape)

Archivos detectados:
- matches: ./matches_1930_2022.csv
- fifa_2022: ./fifa_ranking_2022-10-06.csv
- fifa_2026: ./fifa_ranking_2026-06-08.csv
- schedule: ./schedule_2026_actualizado.csv
- world_cup: ./world_cup.csv

Datasets cargados:
matches: (964, 44)
fifa_2026: (211, 8)
schedule_2026: (104, 16)
fifa_2022: (211, 7)
world_cup: (22, 9)


## 3. Funciones auxiliares de limpieza

In [9]:
# ============================================================
# 3. FUNCIONES AUXILIARES
# ============================================================

def remove_accents(text):
    if pd.isna(text):
        return text
    text = str(text)
    return "".join(
        c for c in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(c)
    )


def clean_text(text):
    if pd.isna(text):
        return ""
    return str(text).strip()


def standardize_team_name(name):
    name = clean_text(name)
    name = re.sub(r"\s+", " ", name)

    replacements = {
        "USA": "United States",
        "U.S.A.": "United States",
        "US": "United States",
        "United States of America": "United States",
        "IR Iran": "Iran",
        "Iran (Islamic Republic of)": "Iran",
        "Islamic Republic of Iran": "Iran",
        "Korea Republic": "South Korea",
        "Republic of Korea": "South Korea",
        "Korea DPR": "North Korea",
        "Côte d’Ivoire": "Côte d'Ivoire",
        "Cote d'Ivoire": "Côte d'Ivoire",
        "Ivory Coast": "Côte d'Ivoire",
        "Türkiye": "Turkey",
        "Turkiye": "Turkey",
        "DR Congo": "Congo DR",
        "Congo DR": "Congo DR",
        "Bosnia & Herzegovina": "Bosnia and Herzegovina",
        "Bosnia-Herzegovina": "Bosnia and Herzegovina"
    }

    return replacements.get(name, name)


def find_col(df, candidates, required=True, label="columna"):
    cols = list(df.columns)
    lower_map = {str(c).lower().strip(): c for c in cols}

    for cand in candidates:
        if cand in cols:
            return cand
        key = str(cand).lower().strip()
        if key in lower_map:
            return lower_map[key]

    if required:
        raise ValueError(
            f"No se encontró {label}. Candidatas: {candidates}. Columnas disponibles: {cols}"
        )
    return None

print("Funciones auxiliares listas.")

Funciones auxiliares listas.


## 4. Preparación de partidos históricos

Se prepara la base histórica para entrenar el modelo de goles esperados.

In [10]:
# ============================================================
# 4. PREPARACIÓN DE PARTIDOS HISTÓRICOS
# ============================================================

home_team_col = find_col(matches, ["home_team", "home", "equipo_local", "local", "team1"], label="equipo local")
away_team_col = find_col(matches, ["away_team", "away", "equipo_visitante", "visitante", "team2"], label="equipo visitante")
home_score_col = find_col(matches, ["home_score", "home_goals", "goles_local", "score_home", "home_goal"], label="goles local")
away_score_col = find_col(matches, ["away_score", "away_goals", "goles_visitante", "score_away", "away_goal"], label="goles visitante")
date_col = find_col(matches, ["date", "fecha", "match_date"], required=False, label="fecha")
year_col = find_col(matches, ["year", "año", "anio", "match_year"], required=False, label="año")
round_col = find_col(matches, ["round", "stage", "fase", "ronda"], required=False, label="ronda")

matches_model = matches.copy()

matches_model["home_team_std"] = matches_model[home_team_col].apply(standardize_team_name)
matches_model["away_team_std"] = matches_model[away_team_col].apply(standardize_team_name)

matches_model["home_goals"] = pd.to_numeric(matches_model[home_score_col], errors="coerce")
matches_model["away_goals"] = pd.to_numeric(matches_model[away_score_col], errors="coerce")

if date_col is not None:
    matches_model["match_date"] = pd.to_datetime(matches_model[date_col], errors="coerce")
else:
    matches_model["match_date"] = pd.NaT

if year_col is not None:
    matches_model["match_year"] = pd.to_numeric(matches_model[year_col], errors="coerce")
else:
    matches_model["match_year"] = matches_model["match_date"].dt.year

matches_model = matches_model.dropna(
    subset=["home_team_std", "away_team_std", "home_goals", "away_goals", "match_year"]
).copy()

matches_model["home_goals"] = matches_model["home_goals"].astype(int)
matches_model["away_goals"] = matches_model["away_goals"].astype(int)
matches_model["match_year"] = matches_model["match_year"].astype(int)

# Resultado para validación simple
matches_model["target"] = np.select(
    [
        matches_model["home_goals"] < matches_model["away_goals"],
        matches_model["home_goals"] == matches_model["away_goals"],
        matches_model["home_goals"] > matches_model["away_goals"]
    ],
    [0, 1, 2]
)

matches_model = matches_model.sort_values(["match_year", "match_date"]).reset_index(drop=True)

print("Partidos históricos preparados:", matches_model.shape)
display(matches_model[["match_year", "home_team_std", "away_team_std", "home_goals", "away_goals", "target"]].head())

Partidos históricos preparados: (964, 51)


,match_year,home_team_std,away_team_std,home_goals,away_goals,target
0,1930,United States,Belgium,3,0,2
1,1930,France,Mexico,4,1,2
2,1930,Yugoslavia,Brazil,2,1,2
3,1930,Romania,Peru,3,1,2
4,1930,Argentina,France,1,0,2


## 5. Ingeniería de variables históricas antes del partido

Las variables se calculan sin usar información del partido actual, para evitar fuga de información.

In [11]:
# ============================================================
# 5. INGENIERÍA DE VARIABLES ANTES DEL PARTIDO
# ============================================================

def empty_stats():
    return {
        "matches": 0,
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "gf": 0,
        "ga": 0,
        "points": 0,
        "recent_points": [],
        "recent_gf": [],
        "recent_ga": []
    }


def stats_to_features(stats):
    m = stats["matches"]
    if m == 0:
        return {
            "matches_before": 0,
            "win_rate_before": 0.0,
            "draw_rate_before": 0.0,
            "loss_rate_before": 0.0,
            "ppm_before": 0.0,
            "gf_avg_before": 0.0,
            "ga_avg_before": 0.0,
            "gd_avg_before": 0.0,
            "recent_ppm_5": 0.0,
            "recent_gf_avg_5": 0.0,
            "recent_ga_avg_5": 0.0,
        }

    recent_points = stats["recent_points"][-5:]
    recent_gf = stats["recent_gf"][-5:]
    recent_ga = stats["recent_ga"][-5:]

    return {
        "matches_before": m,
        "win_rate_before": stats["wins"] / m,
        "draw_rate_before": stats["draws"] / m,
        "loss_rate_before": stats["losses"] / m,
        "ppm_before": stats["points"] / m,
        "gf_avg_before": stats["gf"] / m,
        "ga_avg_before": stats["ga"] / m,
        "gd_avg_before": (stats["gf"] - stats["ga"]) / m,
        "recent_ppm_5": np.mean(recent_points) if recent_points else 0.0,
        "recent_gf_avg_5": np.mean(recent_gf) if recent_gf else 0.0,
        "recent_ga_avg_5": np.mean(recent_ga) if recent_ga else 0.0,
    }


def update_stats(stats, gf, ga):
    stats["matches"] += 1
    stats["gf"] += int(gf)
    stats["ga"] += int(ga)

    if gf > ga:
        pts = 3
        stats["wins"] += 1
    elif gf == ga:
        pts = 1
        stats["draws"] += 1
    else:
        pts = 0
        stats["losses"] += 1

    stats["points"] += pts
    stats["recent_points"].append(pts)
    stats["recent_gf"].append(int(gf))
    stats["recent_ga"].append(int(ga))

team_stats = defaultdict(empty_stats)
rows = []

for idx, row in matches_model.iterrows():
    home = row["home_team_std"]
    away = row["away_team_std"]

    hs = stats_to_features(team_stats[home])
    aas = stats_to_features(team_stats[away])

    feat = {
        "idx": idx,
        "match_year": row["match_year"],
        "home_team_std": home,
        "away_team_std": away,
        "home_goals": row["home_goals"],
        "away_goals": row["away_goals"],
        "target": row["target"],
    }

    for k, v in hs.items():
        feat[f"home_{k}"] = v
    for k, v in aas.items():
        feat[f"away_{k}"] = v

    feat["diff_matches_before"] = hs["matches_before"] - aas["matches_before"]
    feat["diff_win_rate_before"] = hs["win_rate_before"] - aas["win_rate_before"]
    feat["diff_ppm"] = hs["ppm_before"] - aas["ppm_before"]
    feat["diff_gf_avg"] = hs["gf_avg_before"] - aas["gf_avg_before"]
    feat["diff_ga_avg"] = hs["ga_avg_before"] - aas["ga_avg_before"]
    feat["diff_gd_avg"] = hs["gd_avg_before"] - aas["gd_avg_before"]
    feat["diff_recent_ppm_5"] = hs["recent_ppm_5"] - aas["recent_ppm_5"]
    feat["diff_recent_gf_avg_5"] = hs["recent_gf_avg_5"] - aas["recent_gf_avg_5"]
    feat["diff_recent_ga_avg_5"] = hs["recent_ga_avg_5"] - aas["recent_ga_avg_5"]

    rows.append(feat)

    update_stats(team_stats[home], row["home_goals"], row["away_goals"])
    update_stats(team_stats[away], row["away_goals"], row["home_goals"])

features_df = pd.DataFrame(rows)

print("Dataset de modelamiento:", features_df.shape)
display(features_df.head())

Dataset de modelamiento: (964, 38)


,idx,match_year,home_team_std,away_team_std,home_goals,away_goals,target,home_matches_before,home_win_rate_before,home_draw_rate_before,home_loss_rate_before,home_ppm_before,home_gf_avg_before,home_ga_avg_before,home_gd_avg_before,home_recent_ppm_5,home_recent_gf_avg_5,home_recent_ga_avg_5,away_matches_before,away_win_rate_before,away_draw_rate_before,away_loss_rate_before,away_ppm_before,away_gf_avg_before,away_ga_avg_before,away_gd_avg_before,away_recent_ppm_5,away_recent_gf_avg_5,away_recent_ga_avg_5,diff_matches_before,diff_win_rate_before,diff_ppm,diff_gf_avg,diff_ga_avg,diff_gd_avg,diff_recent_ppm_5,diff_recent_gf_avg_5,diff_recent_ga_avg_5
0,0,1930,United States,Belgium,3,0,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1930,France,Mexico,4,1,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,1930,Yugoslavia,Brazil,2,1,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,1930,Romania,Peru,3,1,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,1930,Argentina,France,1,0,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1.0,0.0,0.0,3.0,4.0,1.0,3.0,3.0,4.0,1.0,-1,-1.0,-3.0,-4.0,-1.0,-3.0,-3.0,-4.0,-1.0


## 6. Entrenamiento del modelo Poisson

Se entrenan dos modelos:

- Modelo para goles del equipo local.
- Modelo para goles del equipo visitante.

In [12]:
# ============================================================
# 6. ENTRENAMIENTO DEL MODELO POISSON
# ============================================================

model_features = [
    "home_matches_before",
    "away_matches_before",
    "home_win_rate_before",
    "away_win_rate_before",
    "home_draw_rate_before",
    "away_draw_rate_before",
    "home_loss_rate_before",
    "away_loss_rate_before",
    "home_ppm_before",
    "away_ppm_before",
    "home_gf_avg_before",
    "away_gf_avg_before",
    "home_ga_avg_before",
    "away_ga_avg_before",
    "home_gd_avg_before",
    "away_gd_avg_before",
    "home_recent_ppm_5",
    "away_recent_ppm_5",
    "home_recent_gf_avg_5",
    "away_recent_gf_avg_5",
    "home_recent_ga_avg_5",
    "away_recent_ga_avg_5",
    "diff_matches_before",
    "diff_win_rate_before",
    "diff_ppm",
    "diff_gf_avg",
    "diff_ga_avg",
    "diff_gd_avg",
    "diff_recent_ppm_5",
    "diff_recent_gf_avg_5",
    "diff_recent_ga_avg_5",
]

# Filtrar filas con experiencia previa suficiente para mejorar estabilidad
train_df = features_df.copy()

X = train_df[model_features]
y_home = train_df["home_goals"]
y_away = train_df["away_goals"]

poisson_home = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", PoissonRegressor(alpha=0.05, max_iter=2000))
])

poisson_away = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", PoissonRegressor(alpha=0.05, max_iter=2000))
])

poisson_home.fit(X, y_home)
poisson_away.fit(X, y_away)

pred_home = np.clip(poisson_home.predict(X), 0, 10)
pred_away = np.clip(poisson_away.predict(X), 0, 10)

mae_home = mean_absolute_error(y_home, pred_home)
mae_away = mean_absolute_error(y_away, pred_away)
rmse_home = np.sqrt(mean_squared_error(y_home, pred_home))
rmse_away = np.sqrt(mean_squared_error(y_away, pred_away))

print("Modelo Poisson entrenado correctamente.")
print(f"MAE goles local: {mae_home:.3f}")
print(f"MAE goles visitante: {mae_away:.3f}")
print(f"RMSE goles local: {rmse_home:.3f}")
print(f"RMSE goles visitante: {rmse_away:.3f}")

Modelo Poisson entrenado correctamente.
MAE goles local: 1.128
MAE goles visitante: 0.778
RMSE goles local: 1.465
RMSE goles visitante: 1.029


## 7. Ranking FIFA 2026 y función para partidos futuros

In [13]:
# ============================================================
# 7. RANKING FIFA 2026 Y FUNCIÓN FUTURA
# ============================================================

team_col_fifa = find_col(
    fifa_2026,
    ["team", "country_full", "country", "equipo", "Team", "Country", "name"],
    label="equipo en ranking FIFA 2026"
)

rank_col_fifa = find_col(
    fifa_2026,
    ["rank", "ranking", "Rank", "ranking_fifa_2026"],
    label="ranking FIFA 2026"
)

points_col_fifa = find_col(
    fifa_2026,
    ["total_points", "points", "puntos", "Total Points", "previous_points"],
    required=False,
    label="puntos FIFA 2026"
)

fifa2026_table = fifa_2026.copy()
fifa2026_table["team_std"] = fifa2026_table[team_col_fifa].apply(standardize_team_name)
fifa2026_table["fifa_rank"] = pd.to_numeric(fifa2026_table[rank_col_fifa], errors="coerce")

if points_col_fifa is not None:
    fifa2026_table["fifa_points"] = pd.to_numeric(fifa2026_table[points_col_fifa], errors="coerce")
else:
    fifa2026_table["fifa_points"] = np.nan

fifa2026_table = (
    fifa2026_table
    .dropna(subset=["team_std", "fifa_rank"])
    .drop_duplicates(subset=["team_std"], keep="first")
)

rank_map = fifa2026_table.set_index("team_std")["fifa_rank"].to_dict()
points_map = fifa2026_table.set_index("team_std")["fifa_points"].to_dict()

# Últimas estadísticas históricas disponibles por equipo
latest_team_features = {}
for team, stats in team_stats.items():
    latest_team_features[team] = stats_to_features(stats)


def get_team_future_features(team):
    team = standardize_team_name(team)
    return latest_team_features.get(team, stats_to_features(empty_stats()))


def build_future_features(future_matches):
    df = future_matches.copy()

    if "home_team" not in df.columns or "away_team" not in df.columns:
        raise ValueError("future_matches debe tener columnas home_team y away_team.")

    df["home_team_std"] = df["home_team"].apply(standardize_team_name)
    df["away_team_std"] = df["away_team"].apply(standardize_team_name)

    rows = []

    for _, row in df.iterrows():
        home = row["home_team_std"]
        away = row["away_team_std"]

        hs = get_team_future_features(home)
        aas = get_team_future_features(away)

        feat = row.to_dict()

        for k, v in hs.items():
            feat[f"home_{k}"] = v
        for k, v in aas.items():
            feat[f"away_{k}"] = v

        feat["diff_matches_before"] = hs["matches_before"] - aas["matches_before"]
        feat["diff_win_rate_before"] = hs["win_rate_before"] - aas["win_rate_before"]
        feat["diff_ppm"] = hs["ppm_before"] - aas["ppm_before"]
        feat["diff_gf_avg"] = hs["gf_avg_before"] - aas["gf_avg_before"]
        feat["diff_ga_avg"] = hs["ga_avg_before"] - aas["ga_avg_before"]
        feat["diff_gd_avg"] = hs["gd_avg_before"] - aas["gd_avg_before"]
        feat["diff_recent_ppm_5"] = hs["recent_ppm_5"] - aas["recent_ppm_5"]
        feat["diff_recent_gf_avg_5"] = hs["recent_gf_avg_5"] - aas["recent_gf_avg_5"]
        feat["diff_recent_ga_avg_5"] = hs["recent_ga_avg_5"] - aas["recent_ga_avg_5"]

        feat["home_fifa_rank_2026"] = rank_map.get(home, 999)
        feat["away_fifa_rank_2026"] = rank_map.get(away, 999)
        feat["rank_diff_2026"] = feat["home_fifa_rank_2026"] - feat["away_fifa_rank_2026"]

        feat["home_fifa_points_2026"] = points_map.get(home, np.nan)
        feat["away_fifa_points_2026"] = points_map.get(away, np.nan)

        rows.append(feat)

    out = pd.DataFrame(rows)

    for col in model_features:
        if col not in out.columns:
            out[col] = 0.0
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0.0)

    out["prediccion_valida"] = True
    return out

schedule_home_col = "home_team"
schedule_away_col = "away_team"

print("Ranking FIFA 2026 preparado:", len(rank_map), "equipos")
print("Función build_future_features creada correctamente.")

Ranking FIFA 2026 preparado: 211 equipos
Función build_future_features creada correctamente.


## 8. Funciones de Poisson y lectura de resultados reales

In [14]:
# ============================================================
# 8. FUNCIONES DE POISSON Y LECTURA DE RESULTADOS REALES
# ============================================================

def poisson_probs(lam, max_goals=12):
    lam = max(float(lam), 1e-8)
    probs = np.zeros(max_goals + 1)
    probs[0] = np.exp(-lam)
    for g in range(1, max_goals + 1):
        probs[g] = probs[g - 1] * lam / g
    return probs


def probabilidades_resultado(lambda_home, lambda_away, max_goals=12):
    salida = []
    for lh, la in zip(lambda_home, lambda_away):
        p_home = poisson_probs(lh, max_goals)
        p_away = poisson_probs(la, max_goals)
        matriz = np.outer(p_home, p_away)
        goles_home = np.arange(max_goals + 1)[:, None]
        goles_away = np.arange(max_goals + 1)[None, :]

        p_derrota_local = matriz[goles_home < goles_away].sum()
        p_empate = matriz[goles_home == goles_away].sum()
        p_victoria_local = matriz[goles_home > goles_away].sum()
        total = p_derrota_local + p_empate + p_victoria_local

        if total > 0:
            p_derrota_local /= total
            p_empate /= total
            p_victoria_local /= total

        salida.append([p_derrota_local, p_empate, p_victoria_local])
    return np.array(salida)


def log_poisson(k, lam):
    lam = max(float(lam), 1e-8)
    return -lam + k * log(lam) - lgamma(k + 1)


def marcador_mas_probable(lambda_local, lambda_visitante, max_goles=7):
    mejor_score = -np.inf
    mejor_gl = 0
    mejor_gv = 0
    for gl in range(0, max_goles + 1):
        for gv in range(0, max_goles + 1):
            if gl + gv > 9:
                continue
            score = log_poisson(gl, lambda_local) + log_poisson(gv, lambda_visitante)
            if score > mejor_score:
                mejor_score = score
                mejor_gl = gl
                mejor_gv = gv
    return mejor_gl, mejor_gv, np.exp(mejor_score)


def extraer_resultado(texto_resultado):
    if pd.isna(texto_resultado):
        return pd.Series([np.nan, np.nan, np.nan, np.nan])

    texto = str(texto_resultado).strip()
    if texto == "" or texto.lower() in ["nan", "none", "tbd", "-", "scheduled"]:
        return pd.Series([np.nan, np.nan, np.nan, np.nan])

    marcador = re.search(r"(\d+)\s*-\s*(\d+)", texto)
    if marcador:
        goles_local = int(marcador.group(1))
        goles_visitante = int(marcador.group(2))
    else:
        goles_local = np.nan
        goles_visitante = np.nan

    penales = re.search(r"(?:pen|pens|penales).*?(\d+)\s*-\s*(\d+)", texto, flags=re.IGNORECASE)
    if not penales:
        penales = re.search(r"\((\d+)\s*-\s*(\d+).*?(?:pen|pens|penales)\)", texto, flags=re.IGNORECASE)

    if penales:
        penales_local = int(penales.group(1))
        penales_visitante = int(penales.group(2))
    else:
        penales_local = np.nan
        penales_visitante = np.nan

    return pd.Series([goles_local, goles_visitante, penales_local, penales_visitante])


def definir_ganador_real(fila):
    local = fila["home_team"]
    visitante = fila["away_team"]
    gl = fila["goles_local"]
    gv = fila["goles_visitante"]
    pl = fila["penales_local"]
    pv = fila["penales_visitante"]

    if pd.isna(gl) or pd.isna(gv):
        return pd.Series([np.nan, "Pendiente"])
    if gl > gv:
        return pd.Series([local, "Resultado real: gana en tiempo reglamentario"])
    if gl < gv:
        return pd.Series([visitante, "Resultado real: gana en tiempo reglamentario"])
    if not pd.isna(pl) and not pd.isna(pv):
        if pl > pv:
            return pd.Series([local, "Resultado real: clasifica por penales"])
        if pl < pv:
            return pd.Series([visitante, "Resultado real: clasifica por penales"])
    return pd.Series([np.nan, "Empate sin penales registrados"])

print("Funciones de predicción y resultados listas.")

Funciones de predicción y resultados listas.


## 9. Preparación del fixture actualizado 2026

In [15]:
# ============================================================
# 9. PREPARACIÓN DEL FIXTURE ACTUALIZADO
# ============================================================

fixture = schedule_2026.copy()

fixture_home_col = find_col(fixture, ["home_team", "home", "team1", "equipo_local", "local"], label="equipo local del fixture")
fixture_away_col = find_col(fixture, ["away_team", "away", "team2", "equipo_visitante", "visitante"], label="equipo visitante del fixture")
fixture_matchday_col = find_col(fixture, ["matchday", "round_number", "match_day", "ronda_numero"], required=False, label="matchday")
fixture_status_col = find_col(fixture, ["status", "estado"], required=False, label="estado")
fixture_result_col = find_col(fixture, ["result", "score", "resultado", "marcador"], required=False, label="resultado")
fixture_round_col = find_col(fixture, ["round", "stage", "fase", "ronda"], required=False, label="ronda")
fixture_date_col = find_col(fixture, ["date", "fecha"], required=False, label="fecha")
fixture_time_col = find_col(fixture, ["time", "hora"], required=False, label="hora")

fixture["home_team"] = fixture[fixture_home_col].apply(standardize_team_name)
fixture["away_team"] = fixture[fixture_away_col].apply(standardize_team_name)

if fixture_matchday_col is not None:
    fixture["matchday"] = pd.to_numeric(fixture[fixture_matchday_col], errors="coerce")
else:
    fixture["matchday"] = np.nan

if fixture_status_col is not None:
    fixture["status"] = fixture[fixture_status_col].astype(str).str.strip()
else:
    fixture["status"] = ""

if fixture_result_col is not None:
    fixture["result"] = fixture[fixture_result_col].astype(str).str.strip()
else:
    fixture["result"] = ""

if fixture_round_col is not None:
    fixture["round_text"] = fixture[fixture_round_col].astype(str).str.strip()
else:
    fixture["round_text"] = ""

if fixture_date_col is not None:
    fixture["date"] = fixture[fixture_date_col]
else:
    fixture["date"] = np.nan

if fixture_time_col is not None:
    fixture["time"] = fixture[fixture_time_col]
else:
    fixture["time"] = np.nan

fixture[["goles_local", "goles_visitante", "penales_local", "penales_visitante"]] = fixture["result"].apply(extraer_resultado)
fixture[["ganador_real", "criterio_real"]] = fixture.apply(definir_ganador_real, axis=1)

print("Fixture actualizado preparado:", fixture.shape)
print("Columnas finales usadas:")
print(["matchday", "home_team", "away_team", "status", "result", "round_text"])

print("\nCantidad de partidos por matchday:")
display(fixture["matchday"].value_counts().sort_index())

Fixture actualizado preparado: (104, 18)
Columnas finales usadas:
['matchday', 'home_team', 'away_team', 'status', 'result', 'round_text']

Cantidad de partidos por matchday:


,count
matchday,
1.0,24
2.0,24
3.0,24
16.0,8
32.0,16


## 10. Dieciseisavos reales desde el CSV actualizado

Esta tabla se usa como respaldo para demostrar que no se están inventando los clasificados a octavos.

In [16]:
# ============================================================
# 10. DIECISEISAVOS REALES DESDE CSV
# ============================================================

mask_r32 = (fixture["matchday"] == 32)
if not mask_r32.any():
    mask_r32 = fixture["round_text"].str.lower().str.contains("round of 32|dieciseisavos|32", na=False)

dieciseisavos_reales = fixture[mask_r32].copy().reset_index(drop=True)

if dieciseisavos_reales.empty:
    print("No se encontraron dieciseisavos en el CSV actualizado. Se continuará con octavos.")
    clasificados_reales_a_octavos = pd.DataFrame()
else:
    dieciseisavos_reales["numero_partido"] = range(73, 73 + len(dieciseisavos_reales))
    dieciseisavos_reales["partido"] = dieciseisavos_reales["home_team"] + " vs " + dieciseisavos_reales["away_team"]
    dieciseisavos_reales["marcador_real"] = (
        dieciseisavos_reales["home_team"] + " " +
        dieciseisavos_reales["goles_local"].astype("Int64").astype(str) + " - " +
        dieciseisavos_reales["goles_visitante"].astype("Int64").astype(str) + " " +
        dieciseisavos_reales["away_team"]
    )

    columnas_16 = [
        "numero_partido", "date", "time", "home_team", "away_team",
        "partido", "status", "result", "marcador_real", "ganador_real", "criterio_real"
    ]
    columnas_16 = [c for c in columnas_16 if c in dieciseisavos_reales.columns]

    print("33.1 DIECISEISAVOS REALES DESDE EL CSV ACTUALIZADO")
    print("=" * 90)
    display(dieciseisavos_reales[columnas_16])

    clasificados_reales_a_octavos = dieciseisavos_reales[dieciseisavos_reales["ganador_real"].notna()].copy()
    clasificados_reales_a_octavos = clasificados_reales_a_octavos[[
        "numero_partido", "partido", "marcador_real", "ganador_real", "criterio_real"
    ]].rename(columns={
        "ganador_real": "clasificado_octavos",
        "criterio_real": "criterio_clasificacion"
    })

    print("\n33.2 CLASIFICADOS REALES A OCTAVOS")
    print("=" * 90)
    display(clasificados_reales_a_octavos)

33.1 DIECISEISAVOS REALES DESDE EL CSV ACTUALIZADO


,numero_partido,date,time,home_team,away_team,partido,status,result,marcador_real,ganador_real,criterio_real
0,73,2026-06-28,14:00,South Africa,Canada,South Africa vs Canada,Played,0-1,South Africa 0 - 1 Canada,Canada,Resultado real: gana en tiempo reglamentario
1,74,2026-06-29,12:00,Brazil,Japan,Brazil vs Japan,Played,2-1,Brazil 2 - 1 Japan,Brazil,Resultado real: gana en tiempo reglamentario
2,75,2026-06-29,15:30,Germany,Paraguay,Germany vs Paraguay,Played,1-1 (pens 3-4),Germany 1 - 1 Paraguay,Paraguay,Resultado real: clasifica por penales
3,76,2026-06-29,20:00,Netherlands,Morocco,Netherlands vs Morocco,Played,1-1 (pens 2-3),Netherlands 1 - 1 Morocco,Morocco,Resultado real: clasifica por penales
4,77,2026-06-30,12:00,Côte d'Ivoire,Norway,Côte d'Ivoire vs Norway,Played,1-2,Côte d'Ivoire 1 - 2 Norway,Norway,Resultado real: gana en tiempo reglamentario
5,78,2026-06-30,16:00,France,Sweden,France vs Sweden,Played,3-0,France 3 - 0 Sweden,France,Resultado real: gana en tiempo reglamentario
6,79,2026-06-30,20:00,Mexico,Ecuador,Mexico vs Ecuador,Played,2-0,Mexico 2 - 0 Ecuador,Mexico,Resultado real: gana en tiempo reglamentario
7,80,2026-07-01,11:00,England,Congo DR,England vs Congo DR,Played,2-1,England 2 - 1 Congo DR,England,Resultado real: gana en tiempo reglamentario
8,81,2026-07-01,15:00,Belgium,Senegal,Belgium vs Senegal,Played,3-2,Belgium 3 - 2 Senegal,Belgium,Resultado real: gana en tiempo reglamentario
9,82,2026-07-01,19:00,United States,Bosnia and Herzegovina,United States vs Bosnia and Herzegovina,Played,2-0,United States 2 - 0 Bosnia and Herzegovina,United States,Resultado real: gana en tiempo reglamentario



33.2 CLASIFICADOS REALES A OCTAVOS


,numero_partido,partido,marcador_real,clasificado_octavos,criterio_clasificacion
0,73,South Africa vs Canada,South Africa 0 - 1 Canada,Canada,Resultado real: gana en tiempo reglamentario
1,74,Brazil vs Japan,Brazil 2 - 1 Japan,Brazil,Resultado real: gana en tiempo reglamentario
2,75,Germany vs Paraguay,Germany 1 - 1 Paraguay,Paraguay,Resultado real: clasifica por penales
3,76,Netherlands vs Morocco,Netherlands 1 - 1 Morocco,Morocco,Resultado real: clasifica por penales
4,77,Côte d'Ivoire vs Norway,Côte d'Ivoire 1 - 2 Norway,Norway,Resultado real: gana en tiempo reglamentario
5,78,France vs Sweden,France 3 - 0 Sweden,France,Resultado real: gana en tiempo reglamentario
6,79,Mexico vs Ecuador,Mexico 2 - 0 Ecuador,Mexico,Resultado real: gana en tiempo reglamentario
7,80,England vs Congo DR,England 2 - 1 Congo DR,England,Resultado real: gana en tiempo reglamentario
8,81,Belgium vs Senegal,Belgium 3 - 2 Senegal,Belgium,Resultado real: gana en tiempo reglamentario
9,82,United States vs Bosnia and Herzegovina,United States 2 - 0 Bosnia and Herzegovina,United States,Resultado real: gana en tiempo reglamentario


## 11. Octavos de final: resultados reales y predicción de pendientes

In [17]:
# ============================================================
# 11. OCTAVOS: RESULTADOS REALES + PREDICCIÓN DE PENDIENTES
# ============================================================

mask_r16 = (fixture["matchday"] == 16)
if not mask_r16.any():
    mask_r16 = fixture["round_text"].str.lower().str.contains("round of 16|octavos|16", na=False)

octavos = fixture[mask_r16].copy().reset_index(drop=True)

if octavos.empty:
    raise ValueError(
        "No se encontraron partidos de octavos. Revisa que el CSV tenga matchday == 16 o round = Round Of 16."
    )

octavos["numero_partido"] = range(89, 89 + len(octavos))
octavos["partido"] = octavos["home_team"] + " vs " + octavos["away_team"]

print("33.3 OCTAVOS DEL CSV ACTUALIZADO")
print("=" * 90)
display(octavos[["numero_partido", "date", "time", "home_team", "away_team", "partido", "status", "result", "ganador_real", "criterio_real"]])

# Tabla final
respuesta_3_octavos = octavos.copy()
respuesta_3_octavos["origen_resultado"] = np.where(
    respuesta_3_octavos["ganador_real"].notna(),
    "Resultado real del CSV",
    "Predicción del modelo"
)

respuesta_3_octavos["goles_local_final"] = respuesta_3_octavos["goles_local"]
respuesta_3_octavos["goles_visitante_final"] = respuesta_3_octavos["goles_visitante"]
respuesta_3_octavos["clasificado_cuartos"] = respuesta_3_octavos["ganador_real"]
respuesta_3_octavos["criterio_clasificacion"] = respuesta_3_octavos["criterio_real"]

for col in [
    "goles_local_esperados", "goles_visitante_esperados", "probabilidad_marcador_estimado",
    "prob_derrota_local", "prob_empate", "prob_victoria_local", "confianza_modelo"
]:
    respuesta_3_octavos[col] = np.nan

pendientes = respuesta_3_octavos[respuesta_3_octavos["ganador_real"].isna()].copy()
print("\nPartidos pendientes por predecir:", len(pendientes))

if len(pendientes) > 0:
    future_matches = pd.DataFrame({
        "home_team": pendientes["home_team"].values,
        "away_team": pendientes["away_team"].values,
        "date": pendientes["date"].values if "date" in pendientes.columns else np.nan,
        "time": pendientes["time"].values if "time" in pendientes.columns else np.nan,
        "round": "Octavos de final",
        "year": 2026
    })

    features_octavos = build_future_features(future_matches)
    X_octavos = features_octavos[model_features].copy()

    lambda_local = np.clip(poisson_home.predict(X_octavos), 0.05, 6.0)
    lambda_visitante = np.clip(poisson_away.predict(X_octavos), 0.05, 6.0)

    # Calibración suave con ranking FIFA 2026.
    # Si el local tiene mejor ranking, away_rank - home_rank será positivo.
    if "home_fifa_rank_2026" in features_octavos.columns and "away_fifa_rank_2026" in features_octavos.columns:
        fuerza_ranking = (
            pd.to_numeric(features_octavos["away_fifa_rank_2026"], errors="coerce").fillna(999).values
            -
            pd.to_numeric(features_octavos["home_fifa_rank_2026"], errors="coerce").fillna(999).values
        ) / 100
        fuerza_ranking = np.clip(fuerza_ranking, -1.50, 1.50)
        lambda_local = lambda_local * np.exp(0.12 * fuerza_ranking)
        lambda_visitante = lambda_visitante * np.exp(-0.12 * fuerza_ranking)
        lambda_local = np.clip(lambda_local, 0.05, 6.0)
        lambda_visitante = np.clip(lambda_visitante, 0.05, 6.0)

    probs = probabilidades_resultado(lambda_local, lambda_visitante, max_goals=12)

    indices = pendientes.index.tolist()

    for i, idx in enumerate(indices):
        gl, gv, prob_marcador = marcador_mas_probable(lambda_local[i], lambda_visitante[i], max_goles=7)

        local = respuesta_3_octavos.loc[idx, "home_team"]
        visitante = respuesta_3_octavos.loc[idx, "away_team"]

        respuesta_3_octavos.loc[idx, "goles_local_final"] = gl
        respuesta_3_octavos.loc[idx, "goles_visitante_final"] = gv
        respuesta_3_octavos.loc[idx, "goles_local_esperados"] = lambda_local[i]
        respuesta_3_octavos.loc[idx, "goles_visitante_esperados"] = lambda_visitante[i]
        respuesta_3_octavos.loc[idx, "probabilidad_marcador_estimado"] = prob_marcador

        respuesta_3_octavos.loc[idx, "prob_derrota_local"] = probs[i, 0]
        respuesta_3_octavos.loc[idx, "prob_empate"] = probs[i, 1]
        respuesta_3_octavos.loc[idx, "prob_victoria_local"] = probs[i, 2]
        respuesta_3_octavos.loc[idx, "confianza_modelo"] = probs[i].max()

        if gl > gv:
            clasificado = local
            criterio = "Predicción: gana en tiempo reglamentario"
        elif gl < gv:
            clasificado = visitante
            criterio = "Predicción: gana en tiempo reglamentario"
        else:
            if probs[i, 2] >= probs[i, 0]:
                clasificado = local
            else:
                clasificado = visitante
            criterio = "Predicción: empate; clasifica por penales según mayor probabilidad"

        respuesta_3_octavos.loc[idx, "clasificado_cuartos"] = clasificado
        respuesta_3_octavos.loc[idx, "criterio_clasificacion"] = criterio
else:
    print("Todos los octavos tienen resultado real en el CSV.")

respuesta_3_octavos["goles_local_final"] = respuesta_3_octavos["goles_local_final"].astype("Int64")
respuesta_3_octavos["goles_visitante_final"] = respuesta_3_octavos["goles_visitante_final"].astype("Int64")

respuesta_3_octavos["marcador_final"] = (
    respuesta_3_octavos["home_team"].astype(str) + " " +
    respuesta_3_octavos["goles_local_final"].astype(str) + " - " +
    respuesta_3_octavos["goles_visitante_final"].astype(str) + " " +
    respuesta_3_octavos["away_team"].astype(str)
)

columnas_salida = [
    "numero_partido", "date", "time", "home_team", "away_team", "partido",
    "status", "result", "origen_resultado", "marcador_final",
    "goles_local_esperados", "goles_visitante_esperados",
    "prob_derrota_local", "prob_empate", "prob_victoria_local", "confianza_modelo",
    "clasificado_cuartos", "criterio_clasificacion"
]
columnas_salida = [c for c in columnas_salida if c in respuesta_3_octavos.columns]

respuesta_3_octavos_resultados_actualizados = respuesta_3_octavos[columnas_salida].copy()

print("\n33.4 OCTAVOS: RESULTADOS REALES + PREDICCIÓN DE PENDIENTES")
print("=" * 90)
display(respuesta_3_octavos_resultados_actualizados)

33.3 OCTAVOS DEL CSV ACTUALIZADO


,numero_partido,date,time,home_team,away_team,partido,status,result,ganador_real,criterio_real
0,89,2026-07-04,12:00,Canada,Morocco,Canada vs Morocco,Played,0-3,Morocco,Resultado real: gana en tiempo reglamentario
1,90,2026-07-04,16:00,Paraguay,France,Paraguay vs France,Played,0-1,France,Resultado real: gana en tiempo reglamentario
2,91,2026-07-05,15:00,Brazil,Norway,Brazil vs Norway,To be played,nan,NaN,Pendiente
3,92,2026-07-05,19:00,Mexico,England,Mexico vs England,To be played,nan,NaN,Pendiente
4,93,2026-07-06,14:00,Portugal,Spain,Portugal vs Spain,To be played,nan,NaN,Pendiente
5,94,2026-07-06,19:00,United States,Belgium,United States vs Belgium,To be played,nan,NaN,Pendiente
6,95,2026-07-07,11:00,Argentina,Egypt,Argentina vs Egypt,To be played,nan,NaN,Pendiente
7,96,2026-07-07,15:00,Switzerland,Colombia,Switzerland vs Colombia,To be played,nan,NaN,Pendiente



Partidos pendientes por predecir: 6

33.4 OCTAVOS: RESULTADOS REALES + PREDICCIÓN DE PENDIENTES


,numero_partido,date,time,home_team,away_team,partido,status,result,origen_resultado,marcador_final,goles_local_esperados,goles_visitante_esperados,prob_derrota_local,prob_empate,prob_victoria_local,confianza_modelo,clasificado_cuartos,criterio_clasificacion
0,89,2026-07-04,12:00,Canada,Morocco,Canada vs Morocco,Played,0-3,Resultado real del CSV,Canada 0 - 3 Morocco,NaN,NaN,NaN,NaN,NaN,NaN,Morocco,Resultado real: gana en tiempo reglamentario
1,90,2026-07-04,16:00,Paraguay,France,Paraguay vs France,Played,0-1,Resultado real del CSV,Paraguay 0 - 1 France,NaN,NaN,NaN,NaN,NaN,NaN,France,Resultado real: gana en tiempo reglamentario
2,91,2026-07-05,15:00,Brazil,Norway,Brazil vs Norway,To be played,nan,Predicción del modelo,Brazil 1 - 0 Norway,1.989270,0.520692,0.086510,0.189929,0.723561,0.723561,Brazil,Predicción: gana en tiempo reglamentario
3,92,2026-07-05,19:00,Mexico,England,Mexico vs England,To be played,nan,Predicción del modelo,Mexico 0 - 1 England,0.779772,1.640560,0.579041,0.244581,0.176378,0.579041,England,Predicción: gana en tiempo reglamentario
4,93,2026-07-06,14:00,Portugal,Spain,Portugal vs Spain,To be played,nan,Predicción del modelo,Portugal 1 - 1 Spain,1.087208,1.547314,0.479331,0.253782,0.266887,0.479331,Spain,Predicción: empate; clasifica por penales segú...
5,94,2026-07-06,19:00,United States,Belgium,United States vs Belgium,To be played,nan,Predicción del modelo,United States 0 - 1 Belgium,0.904156,1.283956,0.452510,0.285319,0.262171,0.452510,Belgium,Predicción: gana en tiempo reglamentario
6,95,2026-07-07,11:00,Argentina,Egypt,Argentina vs Egypt,To be played,nan,Predicción del modelo,Argentina 2 - 0 Egypt,2.068708,0.614126,0.098420,0.187036,0.714544,0.714544,Argentina,Predicción: gana en tiempo reglamentario
7,96,2026-07-07,15:00,Switzerland,Colombia,Switzerland vs Colombia,To be played,nan,Predicción del modelo,Switzerland 1 - 1 Colombia,1.477755,1.077975,0.276178,0.260203,0.463619,0.463619,Switzerland,Predicción: empate; clasifica por penales segú...


## 12. Clasificados a cuartos y exportación de resultados

In [18]:
# ============================================================
# 12. CLASIFICADOS A CUARTOS Y EXPORTACIÓN
# ============================================================

os.makedirs("resultados", exist_ok=True)

respuesta_4_clasificados_a_cuartos = respuesta_3_octavos_resultados_actualizados[[
    "numero_partido", "partido", "marcador_final", "origen_resultado",
    "clasificado_cuartos", "criterio_clasificacion", "confianza_modelo"
]].copy()

print("33.5 CLASIFICADOS A CUARTOS DESDE OCTAVOS")
print("=" * 90)
display(respuesta_4_clasificados_a_cuartos)

# Guardar en carpeta resultados
respuesta_3_octavos_resultados_actualizados.to_csv(
    "resultados/respuesta_3_octavos_resultados_actualizados.csv",
    index=False
)

respuesta_4_clasificados_a_cuartos.to_csv(
    "resultados/respuesta_4_clasificados_a_cuartos_actualizados.csv",
    index=False
)

if not clasificados_reales_a_octavos.empty:
    clasificados_reales_a_octavos.to_csv(
        "resultados/clasificados_reales_a_octavos_desde_csv.csv",
        index=False
    )

if not dieciseisavos_reales.empty:
    dieciseisavos_reales.to_csv(
        "resultados/dieciseisavos_reales_desde_csv_actualizado.csv",
        index=False
    )

# Guardar también en raíz para descarga rápida desde Colab
respuesta_3_octavos_resultados_actualizados.to_csv(
    "respuesta_3_octavos_resultados_actualizados.csv",
    index=False
)

respuesta_4_clasificados_a_cuartos.to_csv(
    "respuesta_4_clasificados_a_cuartos_actualizados.csv",
    index=False
)

print("Archivos generados correctamente:")
print("- resultados/respuesta_3_octavos_resultados_actualizados.csv")
print("- resultados/respuesta_4_clasificados_a_cuartos_actualizados.csv")
print("- respuesta_3_octavos_resultados_actualizados.csv")
print("- respuesta_4_clasificados_a_cuartos_actualizados.csv")

33.5 CLASIFICADOS A CUARTOS DESDE OCTAVOS


,numero_partido,partido,marcador_final,origen_resultado,clasificado_cuartos,criterio_clasificacion,confianza_modelo
0,89,Canada vs Morocco,Canada 0 - 3 Morocco,Resultado real del CSV,Morocco,Resultado real: gana en tiempo reglamentario,NaN
1,90,Paraguay vs France,Paraguay 0 - 1 France,Resultado real del CSV,France,Resultado real: gana en tiempo reglamentario,NaN
2,91,Brazil vs Norway,Brazil 1 - 0 Norway,Predicción del modelo,Brazil,Predicción: gana en tiempo reglamentario,0.723561
3,92,Mexico vs England,Mexico 0 - 1 England,Predicción del modelo,England,Predicción: gana en tiempo reglamentario,0.579041
4,93,Portugal vs Spain,Portugal 1 - 1 Spain,Predicción del modelo,Spain,Predicción: empate; clasifica por penales segú...,0.479331
5,94,United States vs Belgium,United States 0 - 1 Belgium,Predicción del modelo,Belgium,Predicción: gana en tiempo reglamentario,0.452510
6,95,Argentina vs Egypt,Argentina 2 - 0 Egypt,Predicción del modelo,Argentina,Predicción: gana en tiempo reglamentario,0.714544
7,96,Switzerland vs Colombia,Switzerland 1 - 1 Colombia,Predicción del modelo,Switzerland,Predicción: empate; clasifica por penales segú...,0.463619


Archivos generados correctamente:
- resultados/respuesta_3_octavos_resultados_actualizados.csv
- resultados/respuesta_4_clasificados_a_cuartos_actualizados.csv
- respuesta_3_octavos_resultados_actualizados.csv
- respuesta_4_clasificados_a_cuartos_actualizados.csv


## 13. Resumen final para exposición

In [19]:
# ============================================================
# 13. RESUMEN FINAL
# ============================================================

print("RESUMEN FINAL PARA EXPOSICIÓN")
print("=" * 90)
print("1. Se entrenó un modelo Poisson con partidos históricos.")
print("2. Se usó ranking FIFA 2026 como referencia actual.")
print("3. Se cargó el fixture actualizado del Mundial 2026.")
print("4. Los partidos de octavos con resultado real fueron respetados.")
print("5. Solo los partidos pendientes fueron predichos por el modelo.")
print("6. La tabla principal para presentar es: respuesta_3_octavos_resultados_actualizados.csv")

print("\nClasificados a cuartos:")
for i, row in respuesta_4_clasificados_a_cuartos.iterrows():
    print(f"{int(row['numero_partido'])}. {row['clasificado_cuartos']} | {row['marcador_final']} | {row['origen_resultado']}")

RESUMEN FINAL PARA EXPOSICIÓN
1. Se entrenó un modelo Poisson con partidos históricos.
2. Se usó ranking FIFA 2026 como referencia actual.
3. Se cargó el fixture actualizado del Mundial 2026.
4. Los partidos de octavos con resultado real fueron respetados.
5. Solo los partidos pendientes fueron predichos por el modelo.
6. La tabla principal para presentar es: respuesta_3_octavos_resultados_actualizados.csv

Clasificados a cuartos:
89. Morocco | Canada 0 - 3 Morocco | Resultado real del CSV
90. France | Paraguay 0 - 1 France | Resultado real del CSV
91. Brazil | Brazil 1 - 0 Norway | Predicción del modelo
92. England | Mexico 0 - 1 England | Predicción del modelo
93. Spain | Portugal 1 - 1 Spain | Predicción del modelo
94. Belgium | United States 0 - 1 Belgium | Predicción del modelo
95. Argentina | Argentina 2 - 0 Egypt | Predicción del modelo
96. Switzerland | Switzerland 1 - 1 Colombia | Predicción del modelo


# Conclusión

El modelo permite estimar los partidos pendientes de octavos de final usando información histórica y ranking FIFA. La mejora principal es que no se predicen partidos que ya tienen resultado real; esos resultados se respetan directamente desde el archivo actualizado.